In [1]:
import cv2
import numpy as np
import random

robu_row = 7
robu_col = 0

obstacles = [
    (1, 1),
    (4, 1),
    (2, 3),
    (5,3),
    (5,4),
    (5,5)
]
starting_point=(0, 350)
ending_point =(350, 0)
action =0
Q = np.zeros((64, 4))



def reset(ep_num): # Ab reset episode number lega
    global robu_row, robu_col
    
    # Episode number ke hisab se 0 se 63 ke beech ek state nikalenge
    state_to_try = ep_num % 64
    
    r = state_to_try // 8
    c = state_to_try % 8
    
    # Agar wo jagah Goal ya Obstacle hai, toh safely (7,0) par bhej do
    if (r, c) == (0, 7) or (r, c) in obstacles:
        robu_row = 7
        robu_col = 0
    else:
        robu_row = r
        robu_col = c
        
    initial_state = robu_row * 8 + robu_col
    return initial_state
    
def step (action):
    global robu_col,robu_row
    if action==0 : robu_row-=1 #up 
    elif action==1 : robu_row+=1 #down
    elif action==2 : robu_col-=1 #left
    elif action==3 : robu_col+=1 #right

    next_state = robu_row*8 + robu_col

    done = False
    if (robu_row,robu_col) ==(0,7):
        reward=100
        done=True
    elif(robu_row,robu_col)in obstacles :
        reward = -100
        done=True
    else :
        reward = -1
        
    return next_state, reward ,done

    
# Hyperparameters
alpha = 0.1    # Learning Rate
gamma = 0.9    # Discount Factor
epsilon = 0.2  # Exploration Rate (20% random moves)
episodes = 200 # Agent kitni baar game khelega

# Valid actions function ko loop se pehle rakhna zaroori hai
def get_valid_actions(r, c):
    actions = [0, 1, 2, 3] # 0:UP, 1:DOWN, 2:LEFT, 3:RIGHT
    if r == 0: actions.remove(0)
    if r == 7: actions.remove(1)
    if c == 0: actions.remove(2)
    if c == 7: actions.remove(3)
    return actions

# Main Training Loop
for episode in range(episodes):
    current_state = reset() # Har episode ke start me agent grid position (7,0) par aayega
    done = False
    
    while not done:
        valid_acts = get_valid_actions(robu_row, robu_col)
        
        if random.uniform(0, 1) < epsilon:
            action = random.choice(valid_acts) # Random move (Explore)
        else:
            # Q-table me se best valid action chunna (Exploit)
            q_values = [Q[current_state, a] for a in valid_acts]
            max_q_idx = np.argmax(q_values)
            action = valid_acts[max_q_idx]
            
        # 2. Environment me step lena
        next_state, reward, done = step(action)
        
        # 3. Bellman Equation (Q-Table Update)
        best_next_action = np.argmax(Q[next_state])
        td_target = reward + gamma * Q[next_state, best_next_action]
        Q[current_state, action] += alpha * (td_target - Q[current_state, action])
        
        # 4. State Update
        current_state = next_state
        
        # 5. OpenCV Visualization (Sirf tabhi jab dekhna ho, training fast karne ke liye)
        canvas = np.zeros((400, 400, 3), dtype=np.uint8)
        # Grid lines
        for i in range(0, 401, 50):
            cv2.line(canvas, (0, i), (400, i), (255, 255, 255), 1)
            cv2.line(canvas, (i, 0), (i, 400), (255, 255, 255), 1)
            
        # Start and Goal
        cv2.rectangle(canvas, (0, 350), (50, 400), (0, 255, 255), -1) # Start
        cv2.rectangle(canvas, (350, 0), (400, 50), (0, 255, 0), -1)   # Goal
        
        # Obstacles
        for obs in obstacles:
            cv2.rectangle(canvas, (obs[1]*50, obs[0]*50), (obs[1]*50+50, obs[0]*50+50), (0, 0, 255), -1)
            
        # Agent current position
        cv2.rectangle(canvas, (robu_col*50, robu_row*50), (robu_col*50+50, robu_row*50+50), (255, 0, 0), -1)
        
        cv2.imshow("Q-Learning", canvas)
        if cv2.waitKey(10) & 0xFF == ord('q'): # 'q' daba kar window band kar sakte ho
            break

cv2.destroyAllWindows()
print("Training Complete! Q-Table is now optimized.")

Training Complete! Q-Table is now optimized.


In [7]:
Q

array([[ 0.00000000e+00, -9.99963565e-01,  0.00000000e+00,
         2.08436479e+00],
       [ 0.00000000e+00, -5.21703100e+01, -9.97261073e-01,
         8.65397400e+00],
       [ 0.00000000e+00, -9.85596735e-01, -7.36734799e-01,
         2.13691408e+01],
       [ 0.00000000e+00, -9.70301176e-01,  3.12241758e+00,
         3.59174020e+01],
       [ 0.00000000e+00, -6.26430520e-01,  1.18772338e+00,
         5.74729645e+01],
       [ 0.00000000e+00,  1.64254621e+00,  5.44017676e+00,
         7.76790463e+01],
       [ 0.00000000e+00,  1.61910965e+01,  1.26871953e+00,
         9.69096846e+01],
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00],
       [-4.99680777e-01, -9.99999607e-01,  0.00000000e+00,
        -8.49905365e+01],
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00],
       [ 2.33265659e+00, -8.12404514e-01, -1.00000000e+01,
        -7.25324473e-01],
       [-6.51321560e-01, -1.00000000e+01, -6.33899820e-01,
      